# 3. Huấn luyện và Đánh giá Mô hình (InceptionTime + BiGRU)

Notebook này được chia thành các bước độc lập, dễ chạy tuần tự (`Run All`) mà không lo ghi đè nhầm checkpoint đã huấn luyện:
1. **Thiết lập môi trường & đường dẫn dữ liệu**
2. **Phân chia tập dữ liệu (`Train` / `Validation` / `Test`) & Cách ly (`Quarantine`)**
3. **Khởi tạo `Dataset` & `DataLoader`**
4. **Khởi tạo mô hình đa phương thức `InceptionBiGRUClassifier`**
5. **Huấn luyện mô hình** *(có cờ `FORCE_RETRAIN` để bỏ qua nếu đã có sẵn checkpoint tốt nhất)*
6. **Nạp Best Checkpoint & Đánh giá trên tập Validation (Fold 9)**
7. **Đánh giá trên tập Test (Fold 10): Trước ($\tau = 0.5$) và Sau khi tối ưu ngưỡng (Threshold Tuning)**

## 1. Thiết lập môi trường & Đường dẫn dữ liệu

In [1]:
from pathlib import Path
import ast
import sys

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

current_dir = Path.cwd().resolve()
project_root = next(
    (path for path in (current_dir, *current_dir.parents)
     if (path / 'data' / 'processed' / 'ptbxl_recordwise_leadwise_zscore_500hz.npy').is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError('Không tìm thấy thư mục gốc chứa data/processed/. Hãy chạy 2_preprocessing.ipynb trước.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data import PTBXLMultiModalDataset, SUPERCLASSES, assert_patient_disjoint
from src.engine import (
    DEFAULT_RECALL_BETAS,
    evaluate_predictions,
    find_optimal_thresholds,
    run_epoch,
)
from src.models import InceptionBiGRUClassifier

data_dir = project_root / 'data'
processed_dir = data_dir / 'processed'
signal_path = processed_dir / 'ptbxl_recordwise_leadwise_zscore_500hz.npy'
signal_index_path = processed_dir / 'ptbxl_recordwise_leadwise_zscore_500hz_index.csv'
clinical_path = processed_dir / 'ptbxl_clinical_features_age_zscore_sex_binary.csv'
checkpoint_dir = project_root / 'models' / 'checkpoints'
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = checkpoint_dir / 'inception_bigru_best.pt'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Project root :', project_root)
print('Device       :', device)
if device.type == 'cuda':
    print('GPU          :', torch.cuda.get_device_name(0))
print('Checkpoint   :', checkpoint_path, f'(exists={checkpoint_path.is_file()})')

Project root : C:\Users\dx\ML in healthcare\Project\roi_loan_nhip_tim_ecg
Device       : cuda
GPU          : NVIDIA GeForce RTX 3050 Laptop GPU
Checkpoint   : C:\Users\dx\ML in healthcare\Project\roi_loan_nhip_tim_ecg\models\checkpoints\inception_bigru_best.pt (exists=True)


## 2. Phân chia tập dữ liệu & Cách ly bệnh nhân ngoại lệ (Quarantine)

- **Fold 1–8**: `Train` | **Fold 9**: `Validation` | **Fold 10**: `Test`
- Loại bỏ toàn bộ bệnh nhân có bản ghi chỉ chứa mã `form/rhythm` không ánh xạ vào 5 superclass khỏi cả 3 tập.
- Kiểm tra nghiêm ngặt **Zero Data Leakage** cấp độ bệnh nhân (`assert_patient_disjoint`).

In [2]:
metadata = pd.read_csv(data_dir / 'ptbxl_database.csv', encoding='latin-1')
scp_statements = pd.read_csv(data_dir / 'scp_statements.csv', index_col=0)
signal_index = pd.read_csv(signal_index_path)
clinical_features = pd.read_csv(clinical_path)

diagnostic = scp_statements.loc[scp_statements['diagnostic'].eq(1.0)]
superclass_lookup = {
    str(code): str(row['diagnostic_class'])
    for code, row in diagnostic.iterrows()
    if pd.notna(row.get('diagnostic_class')) and str(row['diagnostic_class']) in SUPERCLASSES
}
form_rhythm_codes = {
    str(code) for code, row in scp_statements.iterrows()
    if row.get('form') == 1.0 or row.get('rhythm') == 1.0
}

def parse_scp_codes(value):
    if pd.isna(value):
        return {}
    try:
        parsed = ast.literal_eval(value)
    except (TypeError, ValueError, SyntaxError):
        return {}
    return parsed if isinstance(parsed, dict) else {}

def is_form_rhythm_only_unmapped(value):
    codes = set(parse_scp_codes(value))
    return bool(codes) and codes.issubset(form_rhythm_codes) and codes.isdisjoint(superclass_lookup)

review_patient_ids = set(
    metadata.loc[metadata['scp_codes'].map(is_form_rhythm_only_unmapped), 'patient_id']
)
records = signal_index.merge(
    clinical_features, on='ecg_id', how='inner', validate='one_to_one', suffixes=('', '_clinical')
)
eligible = records.loc[~records['patient_id'].isin(review_patient_ids)].copy()

train_records = eligible.loc[eligible['strat_fold'].isin(range(1, 9))].copy()
validation_records = eligible.loc[eligible['strat_fold'].eq(9)].copy()
test_records = eligible.loc[eligible['strat_fold'].eq(10)].copy()
assert_patient_disjoint(train_records, validation_records, test_records)

print(f'Quarantined patients : {len(review_patient_ids):,}')
print(f'Train records        : {len(train_records):,} (folds 1-8)')
print(f'Validation records   : {len(validation_records):,} (fold 9)')
print(f'Test records         : {len(test_records):,} (fold 10)')
print('Patient overlap audit: PASSED')

Quarantined patients : 324
Train records        : 17,005 (folds 1-8)
Validation records   : 2,150 (fold 9)
Test records         : 2,150 (fold 10)
Patient overlap audit: PASSED


## 3. Khởi tạo `Dataset` & `DataLoader`

In [3]:
BATCH_SIZE = 64

train_dataset = PTBXLMultiModalDataset(
    train_records, signal_path, signal_index_path, clinical_path, expected_folds=set(range(1, 9))
)
validation_dataset = PTBXLMultiModalDataset(
    validation_records, signal_path, signal_index_path, clinical_path, expected_folds={9}
)
test_dataset = PTBXLMultiModalDataset(
    test_records, signal_path, signal_index_path, clinical_path, expected_folds={10}
)

loader_options = {
    'batch_size': BATCH_SIZE,
    'num_workers': 0,
    'pin_memory': device.type == 'cuda',
}
train_loader = DataLoader(train_dataset, shuffle=True, **loader_options)
validation_loader = DataLoader(validation_dataset, shuffle=False, **loader_options)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_options)

sample_batch = next(iter(validation_loader))
print('ECG batch shape     :', tuple(sample_batch['ecg'].shape), sample_batch['ecg'].dtype)
print('Clinical batch shape:', tuple(sample_batch['clinical'].shape), '| sample:', sample_batch['clinical'][0].tolist())
print('Target batch shape  :', tuple(sample_batch['target'].shape), '| classes:', SUPERCLASSES)

ECG batch shape     : (64, 12, 5000) torch.float32
Clinical batch shape: (64, 2) | sample: [-0.6826261878013611, 0.0]
Target batch shape  : (64, 5) | classes: ('NORM', 'MI', 'STTC', 'CD', 'HYP')


## 4. Khởi tạo mô hình `InceptionBiGRUClassifier`

Khởi tạo mô hình với cấu hình chuẩn (`branch_channels=32`, `gru_hidden_size=96`, `gru_layers=2`) và kiểm tra forward pass.

In [4]:
model = InceptionBiGRUClassifier().to(device)
parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {parameter_count:,}')

model.eval()
with torch.inference_mode():
    sample_logits = model(sample_batch['ecg'].to(device), sample_batch['clinical'].to(device))
    sample_loss = torch.nn.BCEWithLogitsLoss()(sample_logits, sample_batch['target'].to(device))
print('Forward check       : logits shape =', tuple(sample_logits.shape), f'| initial BCE loss = {float(sample_loss):.4f}')

Trainable parameters: 905,158
Forward check       : logits shape = (64, 5) | initial BCE loss = 0.8398


## 5. Huấn luyện mô hình (Training Loop)

- Mặc định `FORCE_RETRAIN = False`: Nếu tệp checkpoint `models/checkpoints/inception_bigru_best.pt` **đã tồn tại**, cell này sẽ bỏ qua bước train để tiết kiệm thời gian và giữ nguyên trọng số đã huấn luyện.
- Đổi `FORCE_RETRAIN = True` nếu bạn muốn huấn luyện lại từ đầu trên GPU.

In [5]:
FORCE_RETRAIN = False
EPOCHS = 14
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

if checkpoint_path.is_file() and not FORCE_RETRAIN:
    print(f'Đã tìm thấy checkpoint tại: {checkpoint_path}')
    print('Bỏ qua bước huấn luyện (đặt FORCE_RETRAIN = True nếu muốn train lại từ đầu).')
else:
    if device.type != 'cuda':
        raise RuntimeError('Huấn luyện đầy đủ yêu cầu GPU (CUDA). Hãy kiểm tra torch.cuda.is_available().')

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    best_validation_auprc = float('-inf')
    best_epoch = None

    for epoch in range(1, EPOCHS + 1):
        train_metrics = run_epoch(model, train_loader, device, optimizer=optimizer, amp=True)
        val_metrics = run_epoch(model, validation_loader, device, amp=True)
        score = val_metrics['macro_auprc']
        print(
            f'Epoch {epoch:02d}/{EPOCHS} | '
            f'train loss: {train_metrics["loss"]:.4f} | '
            f'val loss: {val_metrics["loss"]:.4f} | '
            f'val macro AUPRC: {score:.6f}'
        )
        if score is not None and score > best_validation_auprc:
            best_validation_auprc = score
            best_epoch = epoch
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'validation_macro_auprc': score,
                'clinical_feature_order': ['age_z', 'sex'],
                'labels': list(SUPERCLASSES),
            }, checkpoint_path)

    if best_epoch is None:
        raise RuntimeError('Không chọn được checkpoint nào.')
    print(f'\nHoàn tất huấn luyện! Best epoch: {best_epoch} (val macro AUPRC = {best_validation_auprc:.6f})')

Đã tìm thấy checkpoint tại: C:\Users\dx\ML in healthcare\Project\roi_loan_nhip_tim_ecg\models\checkpoints\inception_bigru_best.pt
Bỏ qua bước huấn luyện (đặt FORCE_RETRAIN = True nếu muốn train lại từ đầu).


## 6. Nạp Best Checkpoint & Đánh giá trên tập Validation (Fold 9)

- Nạp checkpoint tốt nhất vào `model` và đánh giá trên tập **Validation (Fold 9)**.
- Tìm ngưỡng tối ưu $F_1$-score ($\beta = 1.0$) trên tập Validation cho từng nhãn (riêng nhãn **`CD` giữ nguyên ngưỡng `0.5`**) và in kết quả **Trước** vs. **Sau** khi tối ưu ngưỡng.

In [6]:
if not checkpoint_path.is_file():
    raise FileNotFoundError(f'Không tìm thấy checkpoint: {checkpoint_path}. Hãy chạy huấn luyện ở mục 5 trước.')

checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
if checkpoint.get('labels') != list(SUPERCLASSES):
    raise ValueError('Thứ tự nhãn trong checkpoint không khớp với SUPERCLASSES.')
if checkpoint.get('clinical_feature_order') != ['age_z', 'sex']:
    raise ValueError('Thứ tự đặc trưng lâm sàng trong checkpoint không khớp với [age_z, sex].')

model.load_state_dict(checkpoint['model_state_dict'])
DEFAULT_THRESHOLD = 0.5
val_outputs = run_epoch(
    model, validation_loader, device, amp=True,
    detailed_metrics=True, threshold=DEFAULT_THRESHOLD, return_predictions=True,
)
print(f'Loaded checkpoint from Epoch {checkpoint["epoch"]} (saved val macro AUPRC = {checkpoint["validation_macro_auprc"]:.4f})')

# 1. Kết quả Validation TRƯỚC khi tối ưu ngưỡng (threshold = 0.5)
val_default_table = pd.DataFrame.from_dict(val_outputs['per_label'], orient='index')
val_default_table.index.name = 'Superclass'
print(f'\n=== [VALIDATION] TRƯỚC KHI TỐI ƯU NGƯỠNG (THRESHOLD CỐ ĐỊNH = {DEFAULT_THRESHOLD:.1f}) ===')
print(val_default_table.to_string(float_format=lambda v: f'{v:.4f}'))
print('Validation Macro Metrics (Trước khi tối ưu ngưỡng):')
for metric in ('macro_auroc', 'macro_auprc', 'macro_precision', 'macro_recall', 'macro_f1', 'macro_accuracy', 'subset_accuracy'):
    print(f'  {metric:<16}: {val_outputs[metric]:.4f}')

# 2. Tìm bộ ngưỡng trên Validation (Kịch bản B: F1, riêng CD giữ nguyên 0.5) & in kết quả SAU khi tối ưu
tuned_thresholds = find_optimal_thresholds(
    val_outputs['targets'],
    val_outputs['probabilities'],
    betas=1.0,
)
tuned_thresholds['CD'] = 0.5

val_tuned_metrics = evaluate_predictions(
    val_outputs['targets'],
    val_outputs['probabilities'],
    loss=val_outputs['loss'],
    detailed_metrics=True,
    threshold=tuned_thresholds,
)
val_tuned_table = pd.DataFrame.from_dict(val_tuned_metrics['per_label'], orient='index')
val_tuned_table.index.name = 'Superclass'
print('\n=== [VALIDATION] SAU KHI TỐI ƯU NGƯỠNG (KỊCH BẢN B, RIÊNG CD = 0.5) ===')
print('Bộ ngưỡng tìm được:', tuned_thresholds)
print(val_tuned_table.to_string(float_format=lambda v: f'{v:.4f}'))
print('Validation Macro Metrics (Sau khi tối ưu ngưỡng):')
for metric in ('macro_auroc', 'macro_auprc', 'macro_precision', 'macro_recall', 'macro_f1', 'macro_accuracy', 'subset_accuracy'):
    print(f'  {metric:<16}: {val_tuned_metrics[metric]:.4f}')

Loaded checkpoint from Epoch 12 (saved val macro AUPRC = 0.7634)

=== [VALIDATION] TRƯỚC KHI TỐI ƯU NGƯỠNG (THRESHOLD CỐ ĐỊNH = 0.5) ===
            auroc  auprc  precision  recall     f1  accuracy
Superclass                                                  
NORM       0.9364 0.9070     0.7938  0.9152 0.8502    0.8567
MI         0.9239 0.8338     0.8817  0.5764 0.6971    0.8735
STTC       0.9250 0.7761     0.7125  0.7702 0.7403    0.8665
CD         0.9333 0.8534     0.7695  0.7742 0.7719    0.8944
HYP        0.8126 0.4468     0.5736  0.2731 0.3700    0.8828
Validation Macro Metrics (Trước khi tối ưu ngưỡng):
  macro_auroc     : 0.9062
  macro_auprc     : 0.7634
  macro_precision : 0.7462
  macro_recall    : 0.6618
  macro_f1        : 0.6859
  macro_accuracy  : 0.8748
  subset_accuracy : 0.6000

=== [VALIDATION] SAU KHI TỐI ƯU NGƯỠNG (KỊCH BẢN B, RIÊNG CD = 0.5) ===
Bộ ngưỡng tìm được: {'NORM': 0.645, 'MI': 0.145, 'STTC': 0.495, 'CD': 0.5, 'HYP': 0.27}
            auroc  auprc  threshol

## 7. Đánh giá trên tập Test (Fold 10): Trước và Sau khi tối ưu ngưỡng

Chạy mô hình một lần trên tập **Test (Fold 10)** và in đối chiếu kết quả:
1. **Trước khi tối ưu ngưỡng:** Dùng ngưỡng mặc định $\tau = 0.5$ cho cả 5 nhãn.
2. **Sau khi tối ưu ngưỡng:** Áp dụng bộ ngưỡng `tuned_thresholds` tìm được từ tập Validation ở mục 6 (riêng `CD = 0.5`).

In [7]:
test_outputs = run_epoch(
    model, test_loader, device, amp=True,
    detailed_metrics=True, threshold=DEFAULT_THRESHOLD, return_predictions=True,
)

# 1. Kết quả Test TRƯỚC khi tối ưu ngưỡng (threshold = 0.5)
test_default_table = pd.DataFrame.from_dict(test_outputs['per_label'], orient='index')
test_default_table.index.name = 'Superclass'
print(f'=== [TEST] TRƯỚC KHI TỐI ƯU NGƯỠNG (THRESHOLD CỐ ĐỊNH = {DEFAULT_THRESHOLD:.1f}) ===')
print(test_default_table.to_string(float_format=lambda v: f'{v:.4f}'))
print('Test Macro Metrics (Trước khi tối ưu ngưỡng):')
for metric in ('macro_auroc', 'macro_auprc', 'macro_precision', 'macro_recall', 'macro_f1', 'macro_accuracy', 'subset_accuracy'):
    print(f'  {metric:<16}: {test_outputs[metric]:.4f}')

# 2. Kết quả Test SAU khi tối ưu ngưỡng (dùng tuned_thresholds từ tập Validation)
test_tuned_metrics = evaluate_predictions(
    test_outputs['targets'],
    test_outputs['probabilities'],
    loss=test_outputs['loss'],
    detailed_metrics=True,
    threshold=tuned_thresholds,
)
test_tuned_table = pd.DataFrame.from_dict(test_tuned_metrics['per_label'], orient='index')
test_tuned_table.index.name = 'Superclass'
print('\n=== [TEST] SAU KHI TỐI ƯU NGƯỠNG (KỊCH BẢN B, RIÊNG CD = 0.5) ===')
print('Bộ ngưỡng áp dụng:', tuned_thresholds)
print(test_tuned_table.to_string(float_format=lambda v: f'{v:.4f}'))
print('Test Macro Metrics (Sau khi tối ưu ngưỡng):')
for metric in ('macro_auroc', 'macro_auprc', 'macro_precision', 'macro_recall', 'macro_f1', 'macro_accuracy', 'subset_accuracy'):
    print(f'  {metric:<16}: {test_tuned_metrics[metric]:.4f}')

=== [TEST] TRƯỚC KHI TỐI ƯU NGƯỠNG (THRESHOLD CỐ ĐỊNH = 0.5) ===
            auroc  auprc  precision  recall     f1  accuracy
Superclass                                                  
NORM       0.9361 0.9115     0.7878  0.9291 0.8526    0.8567
MI         0.9165 0.8096     0.8450  0.5064 0.6333    0.8502
STTC       0.9245 0.8051     0.7278  0.7712 0.7488    0.8749
CD         0.9109 0.8304     0.7454  0.7348 0.7401    0.8814
HYP        0.8172 0.4536     0.5942  0.3118 0.4090    0.8898
Test Macro Metrics (Trước khi tối ưu ngưỡng):
  macro_auroc     : 0.9011
  macro_auprc     : 0.7620
  macro_precision : 0.7400
  macro_recall    : 0.6506
  macro_f1        : 0.6768
  macro_accuracy  : 0.8706
  subset_accuracy : 0.5879

=== [TEST] SAU KHI TỐI ƯU NGƯỠNG (KỊCH BẢN B, RIÊNG CD = 0.5) ===
Bộ ngưỡng áp dụng: {'NORM': 0.645, 'MI': 0.145, 'STTC': 0.495, 'CD': 0.5, 'HYP': 0.27}
            auroc  auprc  threshold  precision  recall     f1  accuracy
Superclass                                     